In [21]:
from typing import Dict, List

import torch
from transformers import (
    AutoModelForTokenClassification,
    AutoTokenizer,
)

from transformers import pipeline



MODEL_NAME = "HUMADEX/german_medical_ner"


In [22]:
def get_device():
    """
    Determine the best available computation device.

    Returns
    -------
    torch.device
        CUDA, Apple MPS, or CPU device.
    """
    if torch.cuda.is_available():
        return torch.device("cuda")

    if torch.backends.mps.is_available():
        return torch.device("mps")

    return torch.device("cpu")

In [25]:
def load_medical_ner(
    model_name: str = MODEL_NAME,
):
    """
    Load the German medical NER pipeline.

    Parameters
    ----------
    model_name : str
        Hugging Face model identifier.

    Returns
    -------
    tuple
        NER pipeline and device identifier.
    """
    device = get_device()

    device_index = 0 if device in {"cuda", "mps"} else -1

    ner_pipeline = pipeline(
        "ner",
        model=model_name,
        aggregation_strategy="simple",
        device=device_index,
    )

    return ner_pipeline, device


def clean_entity_text(
    text: str,
):
    """
    Clean whitespace and WordPiece markers from an entity.

    Parameters
    ----------
    text : str
        Entity text returned by the NER pipeline.

    Returns
    -------
    str
        Cleaned entity text.
    """
    text = text.replace("##", "")
    text = " ".join(text.split())

    return text.strip()


def extract_medical_entities(
    query: str,
    ner_pipeline,
):
    """
    Extract medical entities from a query.

    Parameters
    ----------
    query : str
        German medical query.
    ner_pipeline : TokenClassificationPipeline
        Medical NER pipeline.

    Returns
    -------
    list of dict
        Extracted medical entities.
    """
    predictions = ner_pipeline(query)

    entities = []

    for prediction in predictions:
        start = prediction.get("start")
        end = prediction.get("end")

        if start is not None and end is not None:
            text = query[start:end]
        else:
            text = prediction["word"]

        text = clean_entity_text(text)

        entities.append(
            {
                "text": text,
                "label": prediction["entity_group"],
                "score": float(prediction["score"]),
                "start": start,
                "end": end,
            }
        )

    return entities


def extract_medical_keywords(
    query: str,
    ner_pipeline,
):
    """
    Extract medical keywords grouped by entity type.

    Parameters
    ----------
    query : str
        German medical query.
    ner_pipeline : TokenClassificationPipeline
        Medical NER pipeline.

    Returns
    -------
    dict
        Medical keywords grouped by entity type.
    """
    entities = extract_medical_entities(
        query=query,
        ner_pipeline=ner_pipeline,
    )

    keywords = {
        "PROBLEM": [],
        "TEST": [],
        "TREATMENT": [],
    }

    for entity in entities:
        label = entity["label"]
        text = entity["text"]

        if label in keywords and text:
            keywords[label].append(text)

    return keywords


def main():
    """
    Run medical keyword extraction on an example query.
    """
    ner_pipeline, device = load_medical_ner()

    print(f"Using device: {device}")

    query = (
        "Ich habe seit mehreren Tagen starke Kopfschmerzen "
        "und Schwindel und brauche ein MRT."
    )

    entities = extract_medical_entities(
        query=query,
        ner_pipeline=ner_pipeline,
    )

    print("Entities:")
    for entity in entities:
        print(entity)

    keywords = extract_medical_keywords(
        query=query,
        ner_pipeline=ner_pipeline,
    )

    print("\nKeywords:")
    print(keywords)


if __name__ == "__main__":
    main()


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Using device: cpu
Entities:
{'text': 'starke', 'label': 'PROBLEM', 'score': 0.9553462266921997, 'start': 29, 'end': 35}
{'text': 'Kopfschmerzen', 'label': 'PROBLEM', 'score': 0.9967400431632996, 'start': 36, 'end': 49}
{'text': 'Schwindel', 'label': 'PROBLEM', 'score': 0.9961169958114624, 'start': 54, 'end': 63}
{'text': 'e', 'label': 'TEST', 'score': 0.9992744326591492, 'start': 76, 'end': 77}
{'text': 'in', 'label': 'TEST', 'score': 0.8913436532020569, 'start': 77, 'end': 79}
{'text': 'MRT', 'label': 'TEST', 'score': 0.9892791509628296, 'start': 80, 'end': 83}

Keywords:
{'PROBLEM': ['starke', 'Kopfschmerzen', 'Schwindel'], 'TEST': ['e', 'in', 'MRT'], 'TREATMENT': []}
